# H1 — Pre-Roll Deviation vs P&L Scatter

In [1]:
import sys, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, '.')
sys.path.insert(0, '../notebooks')
from config import (RESULTS_DIR, SEAGATE_DIR, FIGS_DIR, WINDOWS_META,
                    BASELINE_STATS, UPDATED_STATS, WIN_COLORS,
                    TICK, MULT, save_fig)
Path('figures').mkdir(exist_ok=True)


In [2]:
import numpy as np

pre_roll_dev = {}
baseline_gross = {}
for wk, wm in WINDOWS_META.items():
    rk = wm['result_key']
    ts = pd.read_parquet(RESULTS_DIR / rk / 'none' / 'timeseries.parquet')
    ts.index = pd.to_datetime(ts.index)
    d1 = wm['days'][0]
    d1_data = ts[ts.index.date.astype(str) == d1]['dev']
    pre_roll_dev[wk] = d1_data.mean()  # D1 mean as proxy
    baseline_gross[wk] = BASELINE_STATS[wk]['gross']

windows = list(WINDOWS_META.keys())
x_dev = [pre_roll_dev[wk] for wk in windows]
y_pnl = [baseline_gross[wk] for wk in windows]

m, b = np.polyfit(x_dev, y_pnl, 1)
x_fit = np.linspace(min(x_dev), max(x_dev), 100)

fig, ax = plt.subplots(figsize=(8, 6))
for wk, xv, yv in zip(windows, x_dev, y_pnl):
    ax.scatter(xv, yv, color=WIN_COLORS[wk], s=120, zorder=5)
    ax.annotate(wk, (xv, yv), textcoords='offset points', xytext=(6, 4), fontsize=10)
ax.plot(x_fit, m*x_fit+b, ls='--', color='grey', lw=1.2, label=f'OLS: y={m:.1f}x+{b:.1f}')
ax.axhline(0, color='black', lw=0.5)
ax.set_xlabel('D1 Mean Deviation (pts) — pre-roll regime proxy')
ax.set_ylabel('Baseline Gross P&L ($)')
ax.set_title('Pre-Roll Deviation vs Strategy P&L\n(caveat: D1 is first trade day, not true pre-roll)')
ax.legend()
ax.annotate('* D1 used as pre-roll proxy; true pre-roll data not available',
            xy=(0.01, 0.01), xycoords='axes fraction', fontsize=8, color='grey')
save_fig(fig, 'H1_preroll_deviation_scatter.png')


  Saved --> figures/H1_preroll_deviation_scatter.png
